# Score a Single Variant

This tutorial exercises the GenoLeWM scoring and receipt path for one ClinVar-like SNV using a tiny deterministic fixture runtime. The output is fixture-smoke evidence only: it validates API wiring and checksum receipts, not learned model quality or clinical behavior.

In [1]:
from __future__ import annotations

import contextlib
import io
import json
import os
import sys
from pathlib import Path
from tempfile import TemporaryDirectory


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "geno_lewm").exists():
            return candidate
    raise RuntimeError("run this notebook from inside the GenoLeWM checkout")


ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

The fixture creates a manifest-backed local model directory, injects deterministic encoder/action/predictor components, and writes a checksum receipt for the score.

In [2]:
from examples.scoring_fixture import (
    CLINVAR_LIKE_VARIANT,
    REFERENCE_WINDOW,
    build_fixture_runtime,
    make_fixture_model_dir,
)

workdir = TemporaryDirectory()
scratch = Path(workdir.name)
model_dir = make_fixture_model_dir(scratch / "model")
runtime = build_fixture_runtime(model_dir)
receipt_path = scratch / "single_variant.receipt.json"

result = runtime.score_variant(
    CLINVAR_LIKE_VARIANT,
    window=REFERENCE_WINDOW,
    receipt_path=receipt_path,
)
summary = {
    "bucket_id": result.bucket_id,
    "confidence": result.confidence,
    "low_confidence": result.low_confidence,
    "receipt_written": receipt_path.is_file(),
    "sigma_calibrated": round(result.sigma_calibrated, 6),
    "sigma_raw": round(result.sigma_raw, 6),
}
print(json.dumps(summary, sort_keys=True))

{"bucket_id": "other|mid|none", "confidence": 1.0, "low_confidence": false, "receipt_written": true, "sigma_calibrated": 0.494975, "sigma_raw": 0.353553}


Receipt validation recomputes the manifest model id, input commitment, and output commitment from the same reference window and edit.

In [3]:
from examples.scoring_fixture import verify_cli_args
from geno_lewm.cli import verify as verify_cli

verify_output = io.StringIO()
with contextlib.redirect_stdout(verify_output):
    rc = verify_cli.main(
        verify_cli_args(receipt_path, model_dir / "manifest.json", REFERENCE_WINDOW)
    )

print(f"receipt validation exit code: {rc}")
print(f"verifier final line: {verify_output.getvalue().splitlines()[-1]}")
workdir.cleanup()

receipt validation exit code: 0
verifier final line: ok
